# Module 4: Clean FL Baselines


## Stage Goal

Run the clean FL baselines once and save `artifacts/module4_clean_baselines.json`. This notebook keeps training output visible so you can watch each clean algorithm progress. Run `train_v3.ipynb` first so the MobileNetV3 target checkpoint exists.


## 1. Notebook Setup


In [ ]:
from pathlib import Path
import sys

MODULE_DIR = Path.cwd()
if (MODULE_DIR / "4_Adversarial_FL").exists():
    MODULE_DIR = MODULE_DIR / "4_Adversarial_FL"
SRC_DIR = MODULE_DIR / "src"
for path in (MODULE_DIR.parent, SRC_DIR):
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

from IPython.display import display
import matplotlib.pyplot as plt

from attack_notebook_utils import (
    clean_result_from_artifact,
    plot_clean_baselines_summary,
    plot_clean_history,
    prepare_context,
    run_clean_baselines,
)

plt.rcParams.update({"figure.dpi": 120, "axes.grid": False})


## 2. Configuration

Edit this cell to change data, FL, or clean-baseline settings. No YAML config is used.


In [ ]:
BASE_FED_CONFIG = {
    "fraction_clients": 0.2,
    "num_clients": 100,
    "num_rounds": 5,
    "num_epochs": 2,
    "batch_size": 64,
    "global_stepsize": 1.0,
    "local_stepsize": 0.002,
    "criterion": "torch.nn.CrossEntropyLoss",
}

ALGORITHMS = {
    "FedAvg": {
        "fed_config": {**BASE_FED_CONFIG, "global_stepsize": 1.0},
        "optim_config": {},
    },
    "FedAdam": {
        "fed_config": {**BASE_FED_CONFIG, "global_stepsize": 0.001},
        "optim_config": {"beta1": 0.9, "beta2": 0.99, "epsilon": 1e-2},
    },
    "FedAdagrad": {
        "fed_config": {**BASE_FED_CONFIG, "global_stepsize": 0.001},
        "optim_config": {"epsilon": 1e-2},
    },
    "FedYogi": {
        "fed_config": {**BASE_FED_CONFIG, "global_stepsize": 0.001},
        "optim_config": {"beta1": 0.9, "beta2": 0.99, "epsilon": 1e-2},
    },
    "Scaffold": {
        "fed_config": {**BASE_FED_CONFIG, "global_stepsize": 0.1},
        "optim_config": {"c_init": 0.0},
    },
}

CONFIG = {
    "quiet": False,
    "clean_baseline_algorithms": ["FedAvg", "FedAdam", "FedAdagrad", "FedYogi", "Scaffold"],
    "data_config": {
        "dataset_path": "./Data/Imagenette",
        "dataset_name": "Imagenette",
        "non_iid_per": 0,
        "eval_subset": "attack_eval",
        "validation_split": {"enabled": True, "seed": 42, "selection_fraction": 0.5},
    },
    "global_config": {"seed": 42, "device": "cuda"},
    "artifacts": {
        "dir": "artifacts",
        "target_checkpoint": "module4_v3_target.pt",
        "surrogate_checkpoint": "module4_surrogate.pt",
        "clean_baselines": "module4_clean_baselines.json",
    },
    "model_config": {
        "module": "model",
        "name": "MobileNetV3Transfer",
        "kwargs": {"pretrained": False, "num_classes": 10, "dropout": 0.1},
    },
    "algorithms": ALGORITHMS,
    "attack": {
        "seed": 42,
        "malicious_fraction": 0.0,
        "malicious_client_selection": {"mode": "none", "client_ids": []},
        "start_round": 3,
        "attack": {
            "type": "random_noise",
            "target_label": 0,
            "poison_rate": 0.0,
            "step_size": 0.0,
            "criterion": "torch.nn.CrossEntropyLoss",
        },
        "surrogate": {
            "checkpoint": "artifacts/module4_surrogate.pt",
            "pretrained": False,
            "num_classes": 10,
            "finetune_epochs": 0,
        },
    },
}

context = prepare_context(CONFIG)
print(f"Device: {context['global_config']['device']}")
print(f"Target checkpoint: {context['target_checkpoint']}")
print(f"Clean baseline artifact: {context['artifact_dir'] / CONFIG['artifacts']['clean_baselines']}")


## 3. Run Clean Baselines

This runs clean FL for the algorithms used by the attack notebooks and writes one shared artifact. Round-by-round training output is shown below.


In [ ]:
clean_baselines = run_clean_baselines(
    context,
    algorithms=CONFIG["clean_baseline_algorithms"],
)

display(clean_baselines["table"])
print(f"Saved clean baselines to {clean_baselines['path']}")


## 4. Final Clean Baseline Plots


In [ ]:
display(clean_baselines["table"])
plot_clean_baselines_summary(clean_baselines, title="Clean FL baselines")
plot_clean_history(
    clean_result_from_artifact(clean_baselines, "FedAvg"),
    title="Clean FedAvg training curve",
)
print("Attack notebooks read this artifact:")
print(clean_baselines["path"])
